## Assignment 4: MCP + Deep Agents in LangGraph

In this assignment, you will start with MCP tool wiring using the current LangChain MCP adapters API, then move into Deep Agents for multi-step research and coding tasks.

### What you will practice
- Connect MCP servers and use MCP tools from a LangChain agent
- Compose a regular LangGraph workflow around `create_agent` (preprocess -> agent -> postprocess)
- Build and run a basic Deep Agent
- Inspect built-in tools and execution traces
- Add custom subagents for delegation
- Load local skills for reusable workflows
- Return structured outputs
- Configure backends safely for project work
- Enable Arize tracing for end-to-end observability


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(".env")


### Tracing with Arize


In [ ]:
print(os.environ["OPENAI_API_KEY"][:10])
print(os.environ["ARIZE_API_KEY"][:10])
print(os.environ["ARIZE_SPACE_ID"][:10])

from arize.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="assignment4-concepts-walkthrough-1",
)

LangChainInstrumentor(tracer_provider=tracer_provider).instrument()

print("Instrumentation active. Sending traces to Arize Cloud Project")


In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "gpt-5-mini",
    model_provider="openai",
    reasoning_effort="low",
)


## 1) MCP tools before Deep Agents

Start here so you can learn MCP tool usage in isolation before adding Deep Agent orchestration.

In this section, you will:
- Connect one or more MCP servers with `MultiServerMCPClient`
- Load tools with `await client.get_tools()`
- Invoke an MCP tool directly (`tool.ainvoke(...)`) to inspect raw output
- Pass MCP tools to `create_agent(...)` so the model can choose tools

These examples intentionally focus on MCP tools only (no resources/prompts).

Reference: [Model Context Protocol (MCP)](https://docs.langchain.com/oss/python/langchain/mcp)


In [ ]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "fetch": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-server-fetch"],
        }
    }
)

mcp_tools = await mcp_client.get_tools()
print([tool.name for tool in mcp_tools])


In [ ]:
fetch_tool = next(tool for tool in mcp_tools if tool.name == "fetch")

fetch_preview = await fetch_tool.ainvoke(
    {"url": "https://docs.langchain.com/oss/python/langchain/mcp"}
)

print(str(fetch_preview)[:1200])


In [ ]:
mcp_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="You are concise agent. Use MCP tools when they help answer the user.",
)

mcp_result = await mcp_agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Fetch https://docs.langchain.com/oss/python/langchain/mcp and summarize "
                    "the MCP workflow in 4 short bullets."
                ),
            }
        ]
    }
)

print(mcp_result["messages"][-1].content)


### Optional: attach your own local MCP server

Use this section when you want to combine a prebuilt MCP server with a local server you control, using `create_agent`.


In [ ]:
from pathlib import Path

LOCAL_MATH_SERVER = (Path.cwd() / "math_server.py").resolve()

if LOCAL_MATH_SERVER.exists():
    custom_client = MultiServerMCPClient(
        {
            "fetch": {
                "transport": "stdio",
                "command": "uvx",
                "args": ["mcp-server-fetch"],
            },
            "math": {
                "transport": "stdio",
                "command": "uv",
                "args": ["run", str(LOCAL_MATH_SERVER)],
            },
        }
    )

    custom_tools = await custom_client.get_tools()
    custom_agent = create_agent(model=llm, tools=custom_tools)

    custom_result = await custom_agent.ainvoke(
        {"messages": [{"role": "user", "content": "What is (23 + 19) * 7? Use tools."}]}
    )
    print(custom_result["messages"][-1].content)
else:
    print("math_server.py was not found in this directory. Add it beside this notebook to run this example.")


In [ ]:
for tool in custom_tools:
    print(tool)

## 2) `create_agent` can be used as part of a graph

You can use `create_agent(...)` as a node inside a regular LangGraph workflow. This pattern is useful for deep-research style pipelines where you:
- pre-process the request,
- run an agent step,
- post-process the output into notes you can pass downstream.

You can then plug the notes and patterns from the rest of this notebook (skills, delegation, structured output) into the same graph design.


In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain_core.messages import AnyMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import Annotated


@tool
def evaluate_expression(expression: str) -> str:
    """Evaluate a valid math expression and return the numeric result as text."""
    return str(eval(expression))


# This ReAct-style agent is a reusable component we can call from a graph node.
embedded_agent = create_agent(
    model=llm,
    tools=[evaluate_expression],
    system_prompt=(
        "You are a careful assistant. Use evaluate_expression for arithmetic "
        "instead of mental math."
    ),
)


class AgentWorkflowState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    preprocessed_query: str
    final_notes: str


def preprocess_node(state: AgentWorkflowState):
    """Normalize the user request before handing it to the agent node."""
    user_message = state["messages"][-1]
    if isinstance(user_message, dict):
        user_text = str(user_message.get("content", ""))
    elif hasattr(user_message, "content"):
        user_text = str(user_message.content)
    else:
        user_text = str(user_message)

    normalized = (
        "Solve carefully and show the result clearly. "
        f"Question: {user_text}"
    )

    return {
        "preprocessed_query": normalized,
        "messages": [
            {"role": "user", "content": normalized}
        ],
    }


def run_agent_node(state: AgentWorkflowState):
    """Run create_agent inside this graph node."""
    agent_result = embedded_agent.invoke({"messages": state["messages"]})
    return {"messages": agent_result["messages"]}


def postprocess_node(state: AgentWorkflowState):
    """Format the final answer as reusable notes."""
    final_answer = state["messages"][-1].content
    notes = (
        "Search Agent Notes\n"
        f"- Normalized query: {state['preprocessed_query']}\n"
        f"- Agent answer: {final_answer}"
    )
    return {"final_notes": notes}


workflow_builder = StateGraph(AgentWorkflowState)
workflow_builder.add_node("preprocess", preprocess_node)
workflow_builder.add_node("agent", run_agent_node)
workflow_builder.add_node("postprocess", postprocess_node)

workflow_builder.add_edge(START, "preprocess")
workflow_builder.add_edge("preprocess", "agent")
workflow_builder.add_edge("agent", "postprocess")
workflow_builder.add_edge("postprocess", END)

agent_in_graph = workflow_builder.compile()


In [ ]:
from IPython.display import Image, display

display(Image(agent_in_graph.get_graph().draw_mermaid_png()))


In [ ]:
workflow_result = agent_in_graph.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "If a $200 bill is split among 4 people with a 20% tip, how much does each person pay?",
            }
        ]
    }
)

print(workflow_result["final_notes"])


## 3) From `create_agent` to `create_deep_agent`

You will start with a familiar ReAct-style agent, then move to Deep Agents for orchestration-heavy tasks.

- Deep Agents quickstart: [docs.langchain.com/oss/python/deepagents/quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)
- Deep Agents overview: [docs.langchain.com/oss/python/deepagents/overview](https://docs.langchain.com/oss/python/deepagents/overview)


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    "Multiply two integers."
    return a * b

react_agent = create_agent(
    model=llm,
    tools=[multiply],
    system_prompt="You are a concise math tutor. Use tools for arithmetic."
)

react_result = react_agent.invoke({
    "messages": [{"role": "user", "content": "What is 27 * 14?"}]
})

print(react_result["messages"][-1].content)

## 4) First Deep Agent

In this section, you create your first Deep Agent.

`create_deep_agent` adds orchestration features on top of ReAct:
- Built-in planning support (`write_todos`)
- Built-in filesystem/shell tools (`ls`, `read_file`, `edit_file`, `glob`, `grep`, `execute`)
- Built-in delegation (`task`) for subagents

Reference: [Deep Agents quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart)


In [ ]:
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

# External research tool
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
):
    "Run a web search and return Tavily results."
    return tavily_client.search(query=query, max_results=max_results, topic=topic)


deep_agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    system_prompt=(
        "You are an expert research assistant. "
        "Break complex tasks into clear steps and produce concise outputs."
    ),
)

### Visualize and inspect built-in capabilities

Inspect the graph view and tool list so you understand what the agent can do before you invoke it.


In [ ]:
from IPython.display import Image, display

display(Image(deep_agent.get_graph().draw_mermaid_png()))

In [ ]:
tool_node = deep_agent.get_graph().nodes["tools"].data
all_tool_names = sorted(tool_node._tools_by_name.keys())
print(all_tool_names)

## 5) Invoke and inspect trace

You invoke Deep Agents with the same message format as `create_agent`, but now the agent can plan and delegate across steps.


In [ ]:
result = deep_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Research the latest open-source text-to-SQL trends and give me a 5-bullet summary."
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    print(f"\n[{i}] {msg_type}")

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call -> {tc['name']}({tc['args']})")

    if getattr(msg, "content", None):
        preview = str(msg.content)
        print("  Content:", preview[:200] + ("..." if len(preview) > 200 else ""))

## 6) Custom subagents with `SubAgent`

Use named specialist subagents to separate responsibilities such as research, writing, and validation.

Reference: [Deep Agents subagents](https://docs.langchain.com/oss/python/deepagents/subagents)


In [ ]:
from deepagents import SubAgent

news_subagent: SubAgent = {
    "name": "news-researcher",
    "description": "Use this for current-events research and source gathering.",
    "system_prompt": "You are a precise news researcher. Return factual source-backed notes.",
    "tools": [internet_search],
}

structured_writer_subagent: SubAgent = {
    "name": "structured-writer",
    "description": "Use this to convert research notes into polished markdown.",
    "system_prompt": "You write concise, high-signal markdown briefings.",
}

delegating_agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    subagents=[news_subagent, structured_writer_subagent],
    system_prompt="Delegate research and writing tasks, then synthesize for the user."
)

In [ ]:
delegated_result = delegating_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Summarize top AI policy updates this month and format it as a briefing note."
        }
    ]
})

print(delegated_result["messages"][-1].content)

## 7) Skills for reusable workflows

Use skills when you want reusable workflows loaded from local directories.

Reference: [Deep Agents skills](https://docs.langchain.com/oss/python/deepagents/skills)

For this assignment setup, use `LocalShellBackend` so skill instructions can execute local Python scripts when needed.

Local skills folder used in this notebook:
`/Users/aish/projects/problem_first_ai/langgraph_2026/skills`


In [ ]:
from pathlib import Path

from deepagents.backends import LocalShellBackend

PROJECT_ROOT = Path.cwd().resolve()
SKILLS_SOURCE = "/skills"

skills_backend = LocalShellBackend(
    root_dir=PROJECT_ROOT,
    virtual_mode=True,
    inherit_env=False,
)

print("Skills root on disk:", PROJECT_ROOT / "skills")
print("Skill folders:", [p.name for p in (PROJECT_ROOT / "skills").iterdir() if p.is_dir()])

skills_agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    skills=[SKILLS_SOURCE],
    backend=skills_backend,
    system_prompt="Use relevant skills when they improve quality or consistency.",
)

skills_result = skills_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create an assignment briefing note on current text-to-SQL progress and include action items."
        }
    ]
})

print(skills_result["messages"][-1].content)


## 8) Structured output from Deep Agents

In this section, you request structured output by passing a schema class directly to `response_format`.

Behavior to remember:
- The schema is validated by the agent runtime.
- The parsed object is returned in the `structured_response` key of the final state.
- With compatible models/providers, LangChain can use provider-native structured output automatically.

References:
- [Deep Agents customization (structured output)](https://docs.langchain.com/oss/python/deepagents/customization#structured-ouput)
- [LangChain structured output](https://docs.langchain.com/oss/python/langchain/structured-output)


In [ ]:
from pydantic import BaseModel, Field

class ResearchSummary(BaseModel):
    topic: str = Field(description="Topic analyzed")
    key_points: list[str] = Field(description="Top findings")
    confidence: str = Field(description="High, Medium, or Low confidence")

structured_agent = create_deep_agent(
    model=llm,
    tools=[internet_search],
    response_format=ResearchSummary,
    system_prompt="Return structured research summaries."
)

structured_result = structured_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Give me a concise current-state summary of open-source text-to-SQL systems."
        }
    ]
})

summary = structured_result["structured_response"]
print(summary)
print(type(summary))

## 9) Backends: what they are and when to use them

Choose a backend based on where you want file operations and shell execution to run.

### Backend selection guide
- `StateBackend` (default): ephemeral filesystem in LangGraph state, scoped to a thread. Use this for temporary working files.
- `FilesystemBackend`: local disk read/write under a configured root. Use this when files must persist on your machine.
- `LocalShellBackend`: local filesystem plus `execute` for shell commands. Use this when workflows need running scripts/tests.
- `StoreBackend`: durable storage across threads via LangGraph store.
- `CompositeBackend`: route different path prefixes to different backends (for example, ephemeral work + persistent `/memories/`).

### Safety notes
- `LocalShellBackend` is powerful and should be scoped tightly with an explicit `root_dir`.
- Prefer `inherit_env=False` unless commands explicitly need host environment variables.
- For `FilesystemBackend`, prefer `virtual_mode=True` for safer path handling.
- For stronger isolation in production-like environments, use sandbox backends.

References:
- [Deep Agents backends](https://docs.langchain.com/oss/python/deepagents/backends)
- [Deep Agents customization](https://docs.langchain.com/oss/python/deepagents/customization)
- [Deep Agents sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes)


In [ ]:
from deepagents.backends import FilesystemBackend, LocalShellBackend

project_root = "/Users/aish/projects/problem_first_ai/langgraph_2026"

# Files-only backend (no shell command execution)
filesystem_backend = FilesystemBackend(
    root_dir=project_root,
    virtual_mode=True,
)

files_only_agent = create_deep_agent(
    model=llm,
    backend=filesystem_backend,
    system_prompt="You can inspect and edit files, but avoid broad destructive edits.",
)

# Assignment backend: filesystem + shell execution (scoped to project root)
local_shell_backend = LocalShellBackend(
    root_dir=project_root,
    inherit_env=False,
    timeout=90,
)

assignment_agent = create_deep_agent(
    model=llm,
    backend=local_shell_backend,
    system_prompt="You are a careful coding assistant. Make small, verifiable changes."
)

assignment_agent